# Phase 2 — Setup & Token Packing

Encodes the Phase 1 JSONL splits into flat `uint16` token streams for pretraining.

**Why this step:** vocab is 8,000, so a token id fits in 2 bytes. Hindi `train.jsonl`
goes 4.55 GiB -> 0.87 GiB and Nepali 6.87 GiB -> 1.06 GiB. That takes SentencePiece
encoding and Drive I/O out of the training loop entirely — the trainer just memory-maps
the `.bin`.

**Runtime:** CPU is fine (no GPU needed). Expect ~30-50 min total.

No GitHub token needed — the code is rclone-synced to Drive.

Measured 2026-09-02 — the packer should reproduce these (excluding `<eos>`):

| | train | val | test | total |
|:--|--:|--:|--:|--:|
| Hindi | 467,874,360 | 26,127,194 | 26,391,043 | **520,392,597** |
| Nepali | 569,684,621 | 31,676,467 | 31,385,814 | **632,746,902** |


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
!ls -la {DRIVE_BASE}

## 2. Install dependencies

In [ ]:
!pip install -q sentencepiece numpy pyyaml

In [ ]:
import os, multiprocessing, shutil
print('vCPUs      :', multiprocessing.cpu_count())
print('free disk  : %.1f GB' % (shutil.disk_usage('/content').free / 1e9))
print('\nPacking is CPU-bound. Free Colab gives 2 vCPUs, so expect\n'
      '~30-50 min for both languages. pack_tokens.py sizes its worker\n'
      'pool to the machine automatically -- do not pass --workers.')


## 3. Get the code

No GitHub token and no clone. The code is synced to Drive with `rclone` from the
local machine, so it arrives on the same mount you already have. It is copied to
`/content/` before running: Drive FUSE is slow for many small reads and cannot
host `__pycache__` writes cleanly, while `/content/` is local NVMe.

Re-run this cell after any code change to pick up the latest sync.

In [ ]:
import shutil, sys, os

SRC  = f'{DRIVE_BASE}/code'      # rclone-synced from the local repo
WORK = '/content/phase2'

if os.path.exists(WORK):
    shutil.rmtree(WORK)          # always take the fresh copy
shutil.copytree(SRC, WORK)

%cd {WORK}
sys.path.insert(0, WORK)
print('files:', sum(len(f) for _, _, f in os.walk(WORK)))
!ls

## 4. Pack Hindi

Reads `.jsonl` (one document per line), **not** `.txt` — 81.9% of documents contain
internal blank lines, so a `\n\n` split would place `<eos>` at ~2.19x too many
positions and teach the model to end documents mid-article.

In [ ]:
!python -m src.data.pack_tokens \
    --input-dir  {DRIVE_BASE}/hindi/splits \
    --tokenizer  tokenizers/hindi/hindi_bpe_8k.model \
    --output-dir {DRIVE_BASE}/hindi/tokens \
    --language   hindi

## 5. Pack Nepali

In [ ]:
!python -m src.data.pack_tokens \
    --input-dir  {DRIVE_BASE}/nepali/splits \
    --tokenizer  tokenizers/nepali/nepali_bpe_8k.model \
    --output-dir {DRIVE_BASE}/nepali/tokens \
    --language   nepali

## 6. Verify against the measured counts

In [ ]:
import json, numpy as np
from pathlib import Path

EXPECTED = {
    'hindi':  {'train': 467_874_360, 'val': 26_127_194, 'test': 26_391_043},
    'nepali': {'train': 569_684_621, 'val': 31_676_467, 'test': 31_385_814},
}

ok = True
for lang, splits in EXPECTED.items():
    print(f'\n=== {lang} ===')
    for split, expect in splits.items():
        d = Path(f'{DRIVE_BASE}/{lang}/tokens')
        st = json.loads((d / f'{lang}_{split}.json').read_text())
        got, err = st['n_tokens_no_eos'], 100 * (st['n_tokens_no_eos'] - expect) / expect
        arr = np.memmap(d / f'{lang}_{split}.bin', dtype=np.uint16, mode='r')
        good = abs(err) < 0.01 and arr.size == st['n_tokens'] and arr.max() < st['vocab_size']
        ok &= good
        print(f"  {split:5} {got:>13,} tok  ({err:+.4f}%)  "
              f"{st['n_eos']:>9,} <eos>  {st['bin_bytes']/2**30:.2f} GiB  "
              f"max_id={arr.max()}  {'OK' if good else 'FAIL'}")

print('\nALL CHECKS PASSED' if ok else '\nSOME CHECKS FAILED')

---
## 7. Model tests

Runs on CPU in seconds. **Do not launch training until these pass.**

`test_causal_mask.py` is required by the brief (§2.1) and is the one test that
catches a silent bug: a model with a broken mask trains fine and reports an
excellent validation loss, because it is partly copying the answer from the
future rather than predicting it.

In [ ]:
!pip install -q pytest
!cd /content/phase2 && python -m pytest tests/ -v --tb=short

## 8. Parameter count and architecture breakdown

In [ ]:
import sys
sys.path.insert(0, '/content/phase2')
%cd /content/phase2

from src.model.transformer_lm import TransformerLM, TransformerLMConfig

for lang in ('hindi', 'nepali'):
    cfg = TransformerLMConfig.from_yaml(f'{lang}/configs/model.yaml')
    model = TransformerLM(cfg)
    bd = model.param_breakdown()
    print(f"\n=== Model {'H' if lang == 'hindi' else 'L'} ({lang}) ===")
    for k, v in bd.items():
        print(f"  {k:20} {v:>12,}")
    assert bd['total'] == bd['expected'] == 25_379_840, bd
print('\nBoth models: 25,379,840 trainable parameters (weight-tied).')